In [1]:
from IPython.display import HTML

HTML('''<script>
code_show=true; 
function code_toggle() {
 if (code_show){
 $('div.input').hide();
 } else {
 $('div.input').show();
 }
 code_show = !code_show
} 
$( document ).ready(code_toggle);
</script>
<form action="javascript:code_toggle()"><input type="submit" value="Click here to toggle on/off the raw code."></form>''')


# 🌊 SWOT Data Notebook

This Colab notebook automatically installs all needed packages, downloads the data, and runs the full analysis.

📦 Required packages will install automatically  
📁 Data will be downloaded via gdown  
⏱️ Just click "Runtime > Run all" to get started!  
❗ The notebook is self-contained until Exercise 6, where login credentials for CMEMS are required.

<a target="_blank" href="https://colab.research.google.com/github/carocamargo/SLSC_SWOT/blob/main/SWOT_Notebook.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 🧭 Table of Contents

- Set Up
- Data Access
- Exercises
  1. Data Exploration
  2. Region Selection and Quality Flags
  3. Altimetry principles and Corrections
  4. Oceanography from SWOT
  5. Coastal SL from SWOT: comparison with tide gauges
  6. Added value of SWOT: Comparing SWOT with other satellites




# Local copy
If you are running this notebook on Google Colab, make sure you make a copy so you can change your edits. For this, go to File > Save a Copy > choose where you would like to save (on your own github or drive).

# Set Up
In this exercise we will use the following packages:
* ```numpy``` : deadling with arrays
* ``` pandas```: deadling with dataframes
*``` xarray```: dealing with netcdf
*```matplotlib```: plotting
*```os```: listing files in the path
*```gdown```: accessing google drive file
*```cartopy```: plotting maps
* ```cmocean```: colormaps for oceanography (https://matplotlib.org/cmocean/)
* ```copernicusmarine```: access CMEMS data (https://marine.copernicus.eu/) - requires login information

Most of these packages are already pre-installed in Google collab, except for ```cartopy```,```gdown```, ```cmocean``` and ```copernicusmarine``` which we will install with pip.

Note, you can comment out the installation and loading of ```copernicusmarine``` if you wont run the extra part.


If you are not using Colab, then make sure you have all the necessary packages installed.

In [2]:
## are we using colab (true/false)
USING_COLAB = "google.colab" in str(get_ipython())

##1. Install packages necessary
This may take a minute or two to install the system dependencies and compile cartopy.

Note, colab runs in a clean, isolated virtual environment for every notebook session, so there's no need to set up a virtual environment manually.

In [3]:
if USING_COLAB:
  !pip install -q gdown
  !pip install cartopy
  !pip install cmocean
  !pip install copernicusmarine "xarray[io]" "zarr<3" fsspec  # comment out if not running Extra

## 2. Load libraries

In [4]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os
import pandas as pd
import cmocean
import matplotlib.cm as mplcm
import copernicusmarine # comment out if not running Extra
if USING_COLAB:
  import gdown
print("All packages imported successfully!")

ModuleNotFoundError: No module named 'copernicusmarine'

# Accessing SWOT data

I've downloaded some SWOT data locally (see [SWOT_Download.ipynb](https://github.com/carocamargo/SLSC_SWOT/blob/main/SWOT_download.ipynb) for example), and loaded a zip folder into my personal google drive, which I gave access for this exercise.

## Using Colab:
You can access this data with ```gdown```.

When you download and unzip files in Colab (like your .zip with gdown), they are saved in the temporary virtual machine storage (e.g., /content/).

This storage is completely wiped when:

* The Colab session times out (typically after 90 minutes of inactivity),

* The user closes the notebook and returns later,

* The notebook is restarted.

## Local computer
If you are not using Colab, then you can download the data to your local computer(https://drive.google.com/file/d/1js5cvDL8_tqZePIg4hs73Sd2vZxBkJOu/view?usp=drive_link) and unzip it to use it. Don't forget then to change the path to access the data in the cell below.



In [ ]:
# if using collab, donwload the data and unzip it
if USING_COLAB:
  data_zipped = 'https://drive.google.com/file/d/1js5cvDL8_tqZePIg4hs73Sd2vZxBkJOu/view?usp=drive_link'
  file_id = data_zipped.split('/')[5]
  gdown.download(f"https://drive.google.com/uc?id={file_id}", quiet=True)
  !unzip SWOT_data.zip

In [ ]:
# get file list
if USING_COLAB:
  path = '/content/data/'
else:
  # if not on colab, then change path to the local path where the data is
  path = '/Users/carocamargo/Documents/data/SWOT/SLSC/data/'
  # dont forget to change this to your local one
files = sorted(os.listdir(path))
files

Note, if you want to keep the data in your personal drive:

### Step 1: mount your google drive


```
from google.colab import drive
drive.mount('/content/drive')
```

### Step 2: Step 2: Move or copy files to your Drive



```
import shutil

# Save to a persistent folder in your Drive
shutil.copytree('/content/data', '/content/drive/MyDrive/SWOT_data')
```

This copies the unzipped data into your Google Drive at MyDrive/SWOT_data, which **does persist**.





# **Exercises**

The goal of this computer exercise is to get familiar with SWOT L3 Expert ocean data. At the end of this tutorial, you will learn which variables are provided, what are the corrections, how can you plot the data, and how to obtain some oceanographic features of interest.

During the morning lectures you learned about the principles of SWOT and saw some applications. In this tutorial you will get hands-on with SWOT data, learning some basics of how to open the data, how to manipulate it, some basic plots and some applications.

This tutorial uses Python (jupyter notebook) to deal with SWOT. Why?
- Python is open source
- It has several tools that makes it easy to deal with nercdfs (e.g., [xarray](https://docs.xarray.dev/en/stable/))
- The [SWOT Community](https://github.com/SWOT-community) is working mainly on python, making it easier to follow the [examples](https://swot-community.github.io/SWOT-galleries/)
- [AVISO](https://swot-community.github.io/SWOT-galleries/SWOT-Oceanography/ex_aviso_download_swot.html), [CMEMS](https://help.marine.copernicus.eu/en/articles/8287609-copernicus-marine-toolbox-api-open-a-dataset-or-read-a-dataframe-remotely) and [JPL PODAAC](https://podaac.github.io/tutorials/quarto_text/SWOT.html) all have jupyter notebooks and python scripts to directly download data.

That being said, you can you use your favorite programming language, and simply use this tutorial as a basis.

We will work with one swath (pass number 236), which crosses the North Sea, and we will use the science orbit.

## **1. Data exploration.**
Open one cycle only (i.e., only one file), and explore the dataset.



In [ ]:
file = files[0]
ds = xr.open_dataset(path+file)
ds

a.   Which variables does it include?

b. What are the dimensions of the variables?

Most of the variables have ```num_lines``` and ```num_pixels``` as dimensions, which represent the **along-track** and the **across-track** indices, respectively.

c. What is the difference between the sea-surface height anomalies provided?

d. Plot the mean sea surface for the entire pass. Which areas does it cross?

e. How long does the cycle take to complete?

e.	When will this pass be repeated again (i.e., when is the next cycle)?

## **2.  Region Selection and Quality Flags.**

Select only the North sea (lon -12 to 12, lat from 48 to 62 degrees north).

In [ ]:
# define our bounding box
lon_range = -12, 12
lat_range = 48, 62
localbox = [lon_range[0], lon_range[1], lat_range[0], lat_range[1]]

In [ ]:
def _normalized_ds(ds, lon_min, lon_max):
    lon = ds.longitude.values
    lon[lon < lon_min] += 360
    lon[lon > lon_max] -= 360
    ds.longitude.values = lon
    return ds

def subset_ds(ds, lon_range, lat_range,variables=False):
    # adapted from
    # https://swot-community.github.io/SWOT-galleries/SWOT-Oceanography/ex_swot_l3_unsmoothed.html
    swot_ds = ds.copy()
    if variables:
      swot_ds = swot_ds[variables]
    swot_ds.load()

    ds = _normalized_ds(swot_ds.copy(), -180, 180)

    mask = (
        (ds.longitude <= lon_range[1])
        & (ds.longitude >= lon_range[0])
        & (ds.latitude <= lat_range[1])
        & (ds.latitude >= lat_range[0])
    ).compute()

    # if we didnt convert our longitude
    # Build mask to cover the two parts
    # mask = (
    #     ((ds.longitude >= lon_range[0]) | (ds.longitude <= lon_range[1]))  # notice the OR
    #     & (ds.latitude >= lat_range[0])
    #     & (ds.latitude <= lat_range[1])
    # ).compute()

    swot_ds_area = swot_ds.where(mask, drop=True)

    if swot_ds_area.sizes['num_lines'] == 0:
        print(f'Dataset {file} not matching geographical area.')
        return None

    for var in list(swot_ds_area.keys()):
        swot_ds_area[var].encoding = {'zlib':True, 'complevel':5}

    return swot_ds_area


In [ ]:
da = subset_ds(ds, lon_range, lat_range)

a. Plot the ssha (unedited)

b.	Look at the quality_flag variable. Which values do we have flagged? Can we trust the data in the Wadden Sea?

## **3. Altimetry principles and corrections**


The surface topography is determined by radar altimeters by measuring the travel time of an emitted radar pulse from the satellite to the surface of the ocean (very simplistic explanation).

The echo of the radar is known as a waveform. From the waveform, we can deduce several parameters, such as the radar range (used to estimate the surface topography), the backscatter coefficient (which informs us about the roughness of the surface), the significant wave height (estimated from the slope of the waveform) and wind speed (derived from the backscatter coefficient and the SWH).

By knowing the altitude of the satellite and having the travel time, it is possible then to determine the surface topography:

**Surface topography = satellite altitude – (altimeter range + geophysical corrections)**

A set of corrections needs to be applied to altimeter range. We won’t go into all of them, because several are applied at lower processing levels. For example, instrument corrections are applied at the raw data. Some of the main corrections are:
- Corrections for perturbations of the radar wave as it crosses the atmosphere;
- Correction for how the sea state directly affects the radar wave (sea state bias, or electromagnetic bias)
- Tidal corrections (ocean tides (barotropic and internal ones), solid earth, pole tides and loading effects)
- Corrections for how the ocean responds to the atmosphere.

Uncorrected data are available in the Level 2 data products. In the Level 3 data that we are using, these corrections have already been applied. A few of them are provided along with the SSHA for users that may want to reverse the correction and recalculate the corrected SSHA with their own “favorite” version. These are dynamic atmospheric correction (DAC), internal tide and ocean tide.

More general SWOT information can be found on the [DUACS SWOT L3 User handbook](https://www.aviso.altimetry.fr/fileadmin/documents/data/tools/hdbk_duacs_SWOT_L3.pdf).

For more information about altimetry principles:
- [AVISO techniques](https://www.aviso.altimetry.fr/en/techniques/altimetry/principle/pulses-and-waveforms.html#:~:text=Analyses%20from%20waveforms%20over%20heterogeneous,on%20flat%20surfaces%20or%20wetlands)
- [Sentinel Wiki Altimetry processing](https://sentiwiki.copernicus.eu/web/altimetry-processing)
- [Sentinel Wiki Altimetry instruments](https://sentiwiki.copernicus.eu/web/s3-altimetry-instruments#S3-Altimetry-Instrument-Backgrounds)

For more information about altimetry corrections:
- [Coastal altimetry workshop presentation by Marcello Passaro and Paolo Cipollini](https://altimetry.esa.int/caw10/old.esaconferencebureau.com/docs/default-source/17c07-img/5-overview-of-altimetry-corrections5eee.pdf?sfvrsn=2)
- [AVISO corrections](https://www.aviso.altimetry.fr/index.php?id=5159)



a. Look at each of these corrections (dac, internal tide and ocean tide). What do they mean? Plot each of them.

b. Compute the uncorrected SSHA and plot it. How different is it?

c. Look at the *sigma0* variable. What does it mean?

d. Which features can you see from sigma0?

e. Apply quality flags to the sigma0 and compare to the calibrated, edited, and filtered SSHA using subplots.

## **4. Oceanography from SWOT**
This problem was designed by Bjarke Nilsson and Ole Anderson.

The ocean topography can tell us a lot about the ocean surface. Here we will see how SWOT observes some of these features.

The feature oceanographers are most often interested in is the sea surface height anomaly (SSHA), as it reveals many dynamic features of the ocean surface. The SSHA is defined as the difference between the sea surface height (SSH) measurement and the mean sea surface (MSS):

        SSHA = SSH - MSS             (1)

However, some features, such as seamounts, do not appear in the SSHA, but are clearly observed in the full sea surface heights (SSH). In order to save space, the only height measurement provided in the datafile is SSHA. But since the MSS is also provided, one can easily compute the SSH.

Another feature of interest for oceanographers are geostrophic currents. Geostrophic currents are an oceanographic feature directly linked to the ocean topography, as described by the equations for geostrophic currents eastward (u) and northward (v):

      u = -(g/f) * ∂ADT/∂y     and     v =  (g/f) * ∂ADT/∂x     (2)

where:

- g is gravity (can be set as 9.82 m/s²),
- f is the Coriolis parameter,
- ADT is the Absolute Dynamic Topography.

One of the unique things with SWOT is the ability to get two-dimensional observations, allowing us to compute the gradient in both directions from a single pass. With conventional altimetry, you would need several individual passes and compute those gradients at crossover points! This allows us to compute features such as geostrophic currents.

Other features of interest for geodesists, such as gravity disturbances caused by seamounts, are often better visualized by the sea surface slopes (SSS) instead of the sea surface heights (SSH):

      ξ = ∂SSH/∂y,     η = ∂SSH/∂x     (3)
as there is a direct relationship between those features.

The Coriolis parameter is given by:

      f = 2 * Ω * sin(φ)     (4)
where:

- Ω is the rotation rate of the Earth (7.292 × 10⁻⁵ s⁻¹),
- φ is the latitude in radians.

The geostrophic currents arise because the ocean is not at rest at an equipotential surface (the geoid), which is why it depends on the Absolute Dynamic Topography (ADT), defined as the sea surface height above the geoid:

      ADT = SSH - N     (5)
The difference between the MSS and the geoid is the Mean Dynamic Topography (MDT):

      MDT = MSS - N     (6)


a. Compute the SSH and ADT from the variables in the file. Plot them.

b. Based on SSH, compute the slopes at each gridpoint.

c. Our data is aligned on the along- and cross-track directions, and not on cartesian coordinates. Based on the orbital inclination of SWOT, which is 78 degrees, we can estimate a rough orbital angle theta (theta = 180 – (90-swotinclination)). Using a [rotation matrix](https://en.wikipedia.org/wiki/Rotation_matrix), we can then rotate the slopes from the along and cross-track directions to north-east directions.

Compute the orbital angle θ, and convert the Along-Track and Cross-Track slopes into North-South and East-West slopes and plot. What changed?

In [ ]:
# To rotate the system we define the rotation matrix
def R(theta):
    # For the definition see: https://en.wikipedia.org/wiki/Rotation_matrix
    return np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])

# We have to determine the orbit angle theta
# A rough way is to know the inclination of SWOT is about 78 degrees:
theta = 180 - (90-78)
# But this is not entirely accurate.



d. Compute the slopes now based on the ADT (remember to convert it to N-E direction).

e. Compute the Coriolis parameter

In [ ]:

# We compute the coriolis parameter

# Defining the constants
g = 9.82
Omega = 7.282*1e-5


f. Now you have all the ingredients to compute geostrophic currents. Compute geostrophic currents u, v, magnitude and plot it

g. Compare the velocities we computed with the ones provided in the dataset already. Look at both the velocities computed with the filtered and unfiltered SSHA.  
Note that, while they can be used to check if your computation is correct, it might not be exactly identitical due to a different noise filter that has been applied.


# 5. Coastal sea level from SWOT
In this exercise, we will compare SWOT with tide gauges


a. Up to now we have been working with only a single cycle. Open one year of SWOT data, which has been provided to you.

In [ ]:
def drop_large_vars(ds):
    to_drop = [v for v in ['i_num_line', 'i_num_pixel'] if v in ds]
    return ds.drop_vars(to_drop)

In [ ]:
# select one year of data that coincides with the TG data
files2 = ['SWOT_L3_LR_SSH_Expert_001_236_20230729T150350_20230729T155516_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_002_236_20230819T114856_20230819T124022_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_003_236_20230909T083402_20230909T092528_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_004_236_20230930T051905_20230930T061031_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_005_236_20231021T020409_20231021T025535_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_006_236_20231110T224914_20231110T234040_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_007_236_20231201T193420_20231201T202546_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_008_236_20231222T161924_20231222T171050_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_009_236_20240112T130429_20240112T135555_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_010_236_20240202T094935_20240202T104102_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_011_236_20240223T063441_20240223T072607_v2.0.1.nc',
#  'SWOT_L3_LR_SSH_Expert_012_236_20240315T031943_20240315T041109_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_013_236_20240405T000447_20240405T005613_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_014_236_20240425T204953_20240425T214120_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_015_236_20240516T173458_20240516T182624_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_016_236_20240606T142004_20240606T151130_v2.0.1.nc',
 'SWOT_L3_LR_SSH_Expert_017_236_20240627T110507_20240627T115633_v2.0.1.nc',]
len(files2)
files_to_open = [path+file for file in files2]
files_to_open
dm = xr.open_mfdataset(files_to_open,concat_dim='cycle',combine='nested',
    preprocess=drop_large_vars)

b. Load time series from 3 tide gauges in the Dutch Wadden Sea available here: https://drive.google.com/file/d/1cdoe_j6XB4CIM3KoN0txvrRL0XZfkeMR/view?usp=sharin

This data was downloaded from CMEMS: https://data.marine.copernicus.eu/product/INSITU_GLO_PHY_SSH_DISCRETE_MY_013_053/services

After downloading, I combined the three tide gauges into a single netcdf, for simplicity.

In [ ]:
# open TG
filename = 'wadden_sea_TGs.nc'
if USING_COLAB:
    tg_url = 'https://drive.google.com/file/d/1cdoe_j6XB4CIM3KoN0txvrRL0XZfkeMR/view?usp=sharing'
    file_id = tg_url.split('/')[5]
    gdown.download(f"https://drive.google.com/uc?id={file_id}", quiet=True)
    path = '/content/'
    dtg = xr.open_dataset(path+filename)
else:
    path = '/Users/carocamargo/Documents/data/sealevel/'
    dtg = xr.open_dataset(path+filename)

In [ ]:
dtg

c. Now, compare SWOT data with the tide gauge data. For this you will need to match the time series both in space and in time. You should also think about which measurements we should compare (hint: tide gauges are measuring total water levels change). Note, for the purposes of this exercise, we can ignore vertical land motion.

In [ ]:
# zoom-in in the wadden sea
localbox_zoom2 = [3.5,6,52,54]

Tide gauges measure total water level (TWL), that is, it includes tides and atmospheric effects to it. 
So to compare SWOT data to tide gauge, we need to re-add tides and DAC to it, as well as the mean sea-surface height

Plot maps:

Plot time series:

In [ ]:

#%% find the closest cell to each station
def haversine(lon1, lat1, lon2, lat2):
    R = 6371.0  # Earth radius in km
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

Consider a radius around tide gauge

# 6. Comparing SWOT with other satellites

In this exercise, we will compare SWOT with L4 Altimetry and SST

Note, for this part you need to have a marine copernicus user. You can register, for free, here: https://data.marine.copernicus.eu/register

a. From CMEMS, load 1 year of L4 gridded all-sat altimetry product.

In [ ]:
# login to copernicus
from copernicusmarine import login
# login(username="your_username", password="your_password")
# you can either define your login here, or it will ask you to fill in below.

In [ ]:
# Load xarray dataset
duacs_l4 = copernicusmarine.open_dataset(
  dataset_id = "cmems_obs-sl_glo_phy-ssh_nrt_allsat-l4-duacs-0.25deg_P1D",
  start_datetime=date_start,
  end_datetime=date_end,
  # start_datetime='2023-08-28',
  # end_datetime='2023-08-30',

  minimum_longitude = lon_range[0],
  maximum_longitude = lon_range[1],
  minimum_latitude = lat_range[0],
  maximum_latitude = lat_range[1],
  variables = ['sla']
)

b. Compare SWOT data with gridded L4 altimetry data. Which features can be seen with SWOT that couldn't be detected in the L4 conventional altimetry?

c. Zoom-in in the Mediterranean sea (lat range: 32 to 45; lon range: 0 to 15), and overlay SWOT SSHA with IFREMER sea-surface temperature (SST). Do you see any patterns?


In [ ]:
# focus on the mediterranean
# Mediterranean sea
lat_range = 32, 45
lon_range = 0, 15
medbox = [lon_range[0], lon_range[1], lat_range[0], lat_range[1]]

In [ ]:
dm = subset_ds(dm, lon_range, lat_range)


In [ ]:
# Load xarray dataset
sst = copernicusmarine.open_dataset(
  dataset_id = "IFREMER-GLOB-SST-L3-NRT-OBS_FULL_TIME_SERIE",
  start_datetime=date_start,
  end_datetime=date_end,
  minimum_longitude = lon_range[0],
  maximum_longitude = lon_range[1],
  minimum_latitude = lat_range[0],
  maximum_latitude = lat_range[1],
  variables = ['sea_surface_temperature']
)

sst['sst_c'] = sst['sea_surface_temperature'] - 273.15
